# 1. Dataloader

## 1.1 Import

In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# PATHS CONFIG — Chỉnh sửa cho phù hợp với tên Kaggle Dataset của bạn
# ─────────────────────────────────────────────────────────────────────────────
VISION_EMB_DIR = Path("/kaggle/input/datasets/kietkolat/vision-embedding-image-des/vision_embeddings_final/embeddings")
ASR_EMB_DIR    = Path("/kaggle/input/datasets/kietkolat/asr-embedding-dataset/asr_embeddings/embeddings")
OCR_EMB_DIR    = Path("/kaggle/input/datasets/kietkolat/ocr-embedding-dataset/ocr_embeddings/embeddings")
CLAP_EMB_DIR   = Path("/kaggle/input/datasets/kiet0lat/clap-audio-embedding/audio_embeddings/embeddings")

ASR_PRIORS_CSV = "/kaggle/input/datasets/kietkolat/asr-embedding-dataset/asr_embeddings/asr_harm_priors.csv"
OCR_PRIORS_CSV = "/kaggle/input/datasets/kietkolat/ocr-embedding-dataset/ocr_embeddings/ocr_aux_priors.csv"

# Embedding dimensions
VISION_DIM  = 1536
ASR_DIM     = 768
OCR_DIM     = 768
CLAP_DIM    = 512
TABULAR_DIM = 12   # 7 ASR priors + 5 OCR priors

ASR_PRIOR_COLS = [
    "information_harm_signal", "sexual_harm_signal", "psychological_harm_signal",
    "hate_harassment_harm_signal", "clickbait_harm_signal",
    "addictive_harm_signal", "physical_harm_signal",
]
OCR_PRIOR_COLS = [
    "aux_psychological", "aux_hate", "aux_sexual",
    "aux_addictive", "aux_clickbait",
]
HARM_COLS = [
    "information_harm", "sexual_harm", "psychological_harm",
    "hate_harassment_harm", "clickbait_harm", "addictive_harm", "physical_harm",
]


## 1.2. Helper

In [2]:
# ── Cell 2: Helper ────────────────────────────────────────────────────────────
def stem(filename: str) -> str:
    """Normalize filename: bỏ .mp4, bỏ khoảng trắng."""
    return str(filename).strip().removesuffix(".mp4")


def load_npy(path: Path, dim: int) -> np.ndarray:
    """Load .npy hoặc trả về zero vector nếu file không tồn tại."""
    if path.exists():
        return np.load(str(path)).astype(np.float32)
    return np.zeros(dim, dtype=np.float32)

## 1.3 Dataset class

In [3]:
class MultimodalDataset(Dataset):
    """
    Args:
        csv_path : path đến stage1_train.csv, stage2_train.csv, v.v.
        mode     : 'stage1' → label is_harmful [1]
                   'stage2' → label 7 harm cols [7]
    """

    def __init__(self, csv_path: str, mode: str = "stage2"):
        assert mode in ("stage1", "stage2")
        self.mode = mode

        self.df = pd.read_csv(csv_path, dtype={"filename": str})
        self.df["stem"] = self.df["filename"].apply(stem)

        # Load tabular priors vào dict để lookup nhanh
        asr_priors = pd.read_csv(ASR_PRIORS_CSV, dtype={"filename": str, "stem_name": str})
        asr_priors["stem"] = asr_priors["stem_name"].apply(stem)
        self.asr_prior_map = asr_priors.set_index("stem")[ASR_PRIOR_COLS].fillna(0).astype(np.float32)

        ocr_priors = pd.read_csv(OCR_PRIORS_CSV, dtype={"filename": str})
        ocr_priors["stem"] = ocr_priors["filename"].apply(stem)
        self.ocr_prior_map = ocr_priors.set_index("stem")[OCR_PRIOR_COLS].fillna(0).astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        s    = row["stem"]

        # ── Load embeddings (zero vector nếu thiếu) ──────────────────────────
        vision_emb = load_npy(VISION_EMB_DIR / f"{s}.npy",  VISION_DIM)
        asr_emb    = load_npy(ASR_EMB_DIR    / f"{s}.npy",  ASR_DIM)
        ocr_emb    = load_npy(OCR_EMB_DIR    / f"{s}.npy",  OCR_DIM)
        clap_emb   = load_npy(CLAP_EMB_DIR   / f"{s}.npy",  CLAP_DIM)

        # ── Load tabular priors ───────────────────────────────────────────────
        asr_prior = (self.asr_prior_map.loc[s].values
                     if s in self.asr_prior_map.index
                     else np.zeros(7, dtype=np.float32))
        ocr_prior = (self.ocr_prior_map.loc[s].values
                     if s in self.ocr_prior_map.index
                     else np.zeros(5, dtype=np.float32))
        tabular = np.concatenate([asr_prior, ocr_prior])  # [12]

        # ── Label ─────────────────────────────────────────────────────────────
        if self.mode == "stage1":
            label = np.array([row["is_harmful"]], dtype=np.float32)
        else:
            label = row[HARM_COLS].values.astype(np.float32)

        return {
            "vision" : torch.from_numpy(vision_emb),
            "asr"    : torch.from_numpy(asr_emb),
            "ocr"    : torch.from_numpy(ocr_emb),
            "clap"   : torch.from_numpy(clap_emb),
            "tabular": torch.from_numpy(tabular),
            "label"  : torch.from_numpy(label),
        }

## 1.4 Factory functions

In [4]:
def make_loader(csv_path: str, mode: str, batch_size: int,
                shuffle: bool = True, num_workers: int = 2) -> DataLoader:
    dataset = MultimodalDataset(csv_path, mode=mode)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True)

## 1.5 Test

In [5]:
# # ── Cell 5: Smoke test ────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     loader = make_loader(
#         "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage2_train.csv",
#         mode="stage2", batch_size=32
#     )
#     batch = next(iter(loader))
#     print("✅ DataLoader smoke test:")
#     for k, v in batch.items():
#         print(f"  {k:8s}: {tuple(v.shape)}  dtype={v.dtype}")
#     # Expected:
#     # vision  : (32, 1536)
#     # asr     : (32, 768)
#     # ocr     : (32, 768)
#     # clap    : (32, 512)
#     # tabular : (32, 12)
#     # label   : (32, 7)


# 2. Fusion Model

## 2.1 Imports

In [6]:
import torch
import torch.nn as nn

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
VISION_DIM  = 1536
ASR_DIM     = 768
OCR_DIM     = 768
CLAP_DIM    = 512
TABULAR_DIM = 12

CONCAT_DIM  = VISION_DIM + ASR_DIM + OCR_DIM + CLAP_DIM  # 3584
INPUT_DIM   = CONCAT_DIM + TABULAR_DIM                    # 3596


## 2.2 Model
- MLP fusion (baseline)

In [7]:
class FusionMLP(nn.Module):
    """
    Simple Concat + MLP Fusion Model.

    Args:
        out_size  : 1 cho Stage 1 (binary), 7 cho Stage 2 (multi-label)
        dropout   : dropout rate (default 0.3)
        hidden_1  : dim của MLP block 1 (default 1024)
        hidden_2  : dim của MLP block 2 (default 256)
    """

    def __init__(self, out_size: int = 7, dropout: float = 0.3,
                 hidden_1: int = 1024, hidden_2: int = 256):
        super().__init__()
        self.out_size = out_size

        self.mlp = nn.Sequential(
            # Block 1
            nn.Linear(INPUT_DIM, hidden_1),
            nn.LayerNorm(hidden_1),
            nn.GELU(),
            nn.Dropout(dropout),
            # Block 2
            nn.Linear(hidden_1, hidden_2),
            nn.LayerNorm(hidden_2),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Linear(hidden_2, out_size)

        # Weight initialization
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, vision, asr, ocr, clap, tabular):
        """
        Args:
            vision  : (B, 1536)
            asr     : (B, 768)
            ocr     : (B, 768)
            clap    : (B, 512)
            tabular : (B, 12)
        Returns:
            logits  : (B, out_size)  — raw logits, CHƯA qua Sigmoid
        """
        x = torch.cat([vision, asr, ocr, clap, tabular], dim=-1)  # (B, 3596)
        x = self.mlp(x)                                            # (B, 256)
        return self.head(x)                                        # (B, out_size)

    def replace_head(self, new_out_size: int):
        """
        Thay thế head layer để chuyển từ Stage 1 (out=1) sang Stage 2 (out=7).
        Các layer MLP trước đó giữ nguyên weights (warm-start).
        """
        in_features  = self.head.in_features
        self.head    = nn.Linear(in_features, new_out_size)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)
        self.out_size = new_out_size
        print(f"✅ Head replaced: {in_features} → {new_out_size}")

    def freeze_mlp(self):
        """Freeze MLP layers, chỉ train head (dùng đầu Stage 2)."""
        for p in self.mlp.parameters():
            p.requires_grad_(False)
        print("🔒 MLP frozen — chỉ train head")

    def unfreeze_all(self):
        """Unfreeze toàn bộ model (dùng sau vài epoch warm-up của Stage 2)."""
        for p in self.parameters():
            p.requires_grad_(True)
        print("🔓 All layers unfrozen")

## 2.3 Test

In [8]:
# if __name__ == "__main__":
#     B = 4
#     model = FusionMLP(out_size=7)
#     print(model)

#     dummy = {
#         "vision" : torch.randn(B, VISION_DIM),
#         "asr"    : torch.randn(B, ASR_DIM),
#         "ocr"    : torch.randn(B, OCR_DIM),
#         "clap"   : torch.randn(B, CLAP_DIM),
#         "tabular": torch.randn(B, TABULAR_DIM),
#     }
#     out = model(**dummy)
#     print(f"✅ Output shape: {out.shape}")  # Expected: (4, 7)

#     total_params = sum(p.numel() for p in model.parameters())
#     print(f"Total parameters: {total_params:,}")

# 3. Fusion stage 1
- This stage will training binary classification (Harmful or Normal)

In [9]:
!pip install scikit-learn -q

## 3.1 Import

In [10]:
import sys, os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import json
import logging
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, roc_auc_score



logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("/kaggle/working/stage1_run.log")
    ]
)
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
TRAIN_CSV   = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage1_train.csv"
VAL_CSV     = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage1_val.csv"
OUTPUT_DIR  = Path("/kaggle/working/stage1_checkpoints")

BATCH_SIZE  = 128
LR          = 1e-4
WEIGHT_DECAY= 1e-4
EPOCHS      = 20
PATIENCE    = 5          # Early stopping
DROPOUT     = 0.3
THRESHOLD   = 0.5
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {DEVICE}")

2026-07-29 06:42:41,023 INFO Device: cuda


## 3.2 DataLoaders

In [11]:
train_loader = make_loader(TRAIN_CSV, mode="stage1", batch_size=BATCH_SIZE, shuffle=True)
val_loader   = make_loader(VAL_CSV,   mode="stage1", batch_size=BATCH_SIZE, shuffle=False)

# Tính pos_weight: số mẫu negative / số mẫu positive
train_df   = pd.read_csv(TRAIN_CSV)
n_pos      = train_df["is_harmful"].sum()
n_neg      = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
logger.info(f"Train: {len(train_df):,} | Harmful={n_pos:,} | Normal={n_neg:,}")
logger.info(f"pos_weight: {pos_weight.item():.3f}")

2026-07-29 06:42:41,507 INFO Train: 4,400 | Harmful=3,266 | Normal=1,134
2026-07-29 06:42:41,516 INFO pos_weight: 0.347


## 3.3 Model, Loss, Optimizer

In [12]:
model     = FusionMLP(out_size=1, dropout=DROPOUT).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
logger.info(f"Model params: {total_params:,}")

2026-07-29 06:42:44,732 INFO Model params: 3,948,545


## 3.4 Train / Val functions

In [13]:
def run_epoch(loader, model, criterion, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_probs  = []
    all_labels = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc="train" if is_train else "val", leave=False):
            vision  = batch["vision"].to(device)
            asr     = batch["asr"].to(device)
            ocr     = batch["ocr"].to(device)
            clap    = batch["clap"].to(device)
            tabular = batch["tabular"].to(device)
            labels  = batch["label"].to(device)   # (B, 1)

            logits = model(vision, asr, ocr, clap, tabular)  # (B, 1)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            all_probs.extend(probs.flatten().tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())

    avg_loss = total_loss / len(loader.dataset)
    preds    = (np.array(all_probs) >= THRESHOLD).astype(int)
    f1       = f1_score(all_labels, preds, zero_division=0)
    roc_auc  = roc_auc_score(all_labels, all_probs)
    return avg_loss, f1, roc_auc

## 3.5 Traning loop

In [14]:
# best_roc_auc  = 0.0
# patience_cnt  = 0
# history       = []

# logger.info("=" * 60)
# logger.info("🚀 BẮT ĐẦU STAGE 1 TRAINING")
# logger.info("=" * 60)

# for epoch in range(1, EPOCHS + 1):
#     train_loss, train_f1, train_roc = run_epoch(train_loader, model, criterion, optimizer)
#     val_loss,   val_f1,   val_roc   = run_epoch(val_loader,   model, criterion)
#     scheduler.step()

#     logger.info(
#         f"Epoch {epoch:02d}/{EPOCHS} | "
#         f"Train Loss={train_loss:.4f} F1={train_f1:.4f} ROC={train_roc:.4f} | "
#         f"Val   Loss={val_loss:.4f} F1={val_f1:.4f} ROC={val_roc:.4f}"
#     )

#     history.append({
#         "epoch": epoch,
#         "train_loss": train_loss, "train_f1": train_f1, "train_roc_auc": train_roc,
#         "val_loss": val_loss,     "val_f1": val_f1,     "val_roc_auc": val_roc,
#     })

#     # Checkpoint tốt nhất theo Val ROC-AUC
#     if val_roc > best_roc_auc:
#         best_roc_auc = val_roc
#         patience_cnt = 0
#         torch.save({
#             "epoch"     : epoch,
#             "model_state": model.state_dict(),
#             "optimizer_state": optimizer.state_dict(),
#             "val_roc_auc": val_roc,
#             "val_f1"    : val_f1,
#         }, OUTPUT_DIR / "best_model.pt")
#         logger.info(f"  ✅ New best ROC-AUC: {best_roc_auc:.4f} — saved best_model.pt")
#     else:
#         patience_cnt += 1
#         if patience_cnt >= PATIENCE:
#             logger.info(f"  ⏹️  Early stopping at epoch {epoch} (patience={PATIENCE})")
#             break

# # Lưu checkpoint cuối
# torch.save({"epoch": epoch, "model_state": model.state_dict()},
#            OUTPUT_DIR / "last_model.pt")


## 3.6 Save history & report 

In [15]:
# pd.DataFrame(history).to_csv(OUTPUT_DIR / "train_history.csv", index=False)

# logger.info("=" * 60)
# logger.info(f"✅ Stage 1 hoàn thành! Best Val ROC-AUC: {best_roc_auc:.4f}")
# logger.info("📦 NEXT STEPS:")
# logger.info("  1. Save output → Kaggle Dataset 'stage1-checkpoints'")
# logger.info("  2. Attach tới fusion_nb4_stage2_train.py")
# logger.info("=" * 60)


# 4. Fusion Stage 2

## 4.1 Import

In [16]:
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import json
import logging
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import f1_score



logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("/kaggle/working/stage2_run.log")
    ]
)
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
STAGE1_CKPT  = "/kaggle/input/datasets/kietkolat/stage-1-dataset/stage1_checkpoints/best_model.pt"
TRAIN_CSV    = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage2_train.csv"
VAL_CSV      = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage2_val.csv"
OUTPUT_DIR   = Path("/kaggle/working/stage2_checkpoints")

BATCH_SIZE   = 64
LR_FROZEN    = 3e-4   # LR khi MLP frozen (chỉ train head)
LR_UNFROZEN  = 5e-5   # LR sau khi unfreeze toàn bộ
WEIGHT_DECAY = 1e-4
EPOCHS       = 30
FREEZE_EPOCHS= 2      # Số epoch freeze MLP
PATIENCE     = 7
DROPOUT      = 0.3
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {DEVICE}")

2026-07-29 06:42:44,800 INFO Device: cuda


## 4.2 DataLoaders

In [17]:
train_loader = make_loader(TRAIN_CSV, mode="stage2", batch_size=BATCH_SIZE, shuffle=True)
val_loader   = make_loader(VAL_CSV,   mode="stage2", batch_size=BATCH_SIZE, shuffle=False)

# Tính pos_weight riêng cho từng nhãn trong 7 nhãn
train_df  = pd.read_csv(TRAIN_CSV)
n_total   = len(train_df)
pos_weights = []
logger.info("Per-label pos_weight (Stage 2):")
for col in HARM_COLS:
    n_pos = train_df[col].sum()
    n_neg = n_total - n_pos
    pw    = n_neg / max(n_pos, 1)
    pos_weights.append(pw)
    logger.info(f"  {col:<30}: pos={n_pos:4d} neg={n_neg:4d} pw={pw:.2f}")

pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(DEVICE)

2026-07-29 06:42:44,899 INFO Per-label pos_weight (Stage 2):
2026-07-29 06:42:44,901 INFO   information_harm              : pos= 188 neg=3078 pw=16.37
2026-07-29 06:42:44,902 INFO   sexual_harm                   : pos= 373 neg=2893 pw=7.76
2026-07-29 06:42:44,904 INFO   psychological_harm            : pos= 702 neg=2564 pw=3.65
2026-07-29 06:42:44,905 INFO   hate_harassment_harm          : pos= 378 neg=2888 pw=7.64
2026-07-29 06:42:44,906 INFO   clickbait_harm                : pos= 580 neg=2686 pw=4.63
2026-07-29 06:42:44,907 INFO   addictive_harm                : pos= 799 neg=2467 pw=3.09
2026-07-29 06:42:44,908 INFO   physical_harm                 : pos= 757 neg=2509 pw=3.31


## 4.3 Warm-start Model

In [18]:
logger.info(f"Loading Stage 1 checkpoint: {STAGE1_CKPT}")
model = FusionMLP(out_size=1, dropout=DROPOUT).to(DEVICE)
ckpt  = torch.load(STAGE1_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
logger.info(f"Stage 1 loaded (Val ROC-AUC was {ckpt.get('val_roc_auc', '?'):.4f})")

# Thay head từ 1 neuron → 7 neurons
model.replace_head(new_out_size=7)

model = model.to(DEVICE)

# Bắt đầu với MLP frozen
model.freeze_mlp()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_FROZEN, weight_decay=WEIGHT_DECAY
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

2026-07-29 06:42:44,917 INFO Loading Stage 1 checkpoint: /kaggle/input/datasets/kietkolat/stage-1-dataset/stage1_checkpoints/best_model.pt
2026-07-29 06:42:46,035 INFO Stage 1 loaded (Val ROC-AUC was 0.9582)


✅ Head replaced: 256 → 7
🔒 MLP frozen — chỉ train head


## 4.4 Metric helpers

In [19]:
def compute_f1_macro(labels: np.ndarray, probs: np.ndarray,
                     threshold: float = 0.5) -> float:
    preds = (probs >= threshold).astype(int)
    return f1_score(labels, preds, average="macro", zero_division=0)


def find_best_thresholds(labels: np.ndarray, probs: np.ndarray) -> list:
    """Tìm threshold tối ưu cho từng nhãn trên Val set."""
    thresholds = np.arange(0.1, 0.9, 0.05)
    best_ts = []
    for i in range(labels.shape[1]):
        best_t, best_f1 = 0.5, 0.0
        for t in thresholds:
            preds = (probs[:, i] >= t).astype(int)
            f1    = f1_score(labels[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_ts.append(float(round(best_t, 2)))
    return best_ts


## 4.5 Train / Val functions

In [20]:
def run_epoch(loader, model, criterion, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss  = 0.0
    all_probs   = []
    all_labels  = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for batch in tqdm(loader, desc="train" if is_train else "val", leave=False):
            vision  = batch["vision"].to(device)
            asr     = batch["asr"].to(device)
            ocr     = batch["ocr"].to(device)
            clap    = batch["clap"].to(device)
            tabular = batch["tabular"].to(device)
            labels  = batch["label"].to(device)   # (B, 7)

            logits = model(vision, asr, ocr, clap, tabular)  # (B, 7)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    avg_loss   = total_loss / len(loader.dataset)
    all_probs  = np.vstack(all_probs)   # (N, 7)
    all_labels = np.vstack(all_labels)  # (N, 7)
    f1_macro   = compute_f1_macro(all_labels, all_probs)

    return avg_loss, f1_macro, all_probs, all_labels

## 4.6 Training loop

In [21]:
 
# best_f1_macro = 0.0
# patience_cnt  = 0
# history       = []
# scheduler     = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# logger.info("=" * 60)
# logger.info("🚀 BẮT ĐẦU STAGE 2 TRAINING (Warm-start từ Stage 1)")
# logger.info("=" * 60)

# for epoch in range(1, EPOCHS + 1):

#     # Unfreeze sau FREEZE_EPOCHS epoch
#     if epoch == FREEZE_EPOCHS + 1:
#         model.unfreeze_all()
#         optimizer = torch.optim.AdamW(
#             model.parameters(), lr=LR_UNFROZEN, weight_decay=WEIGHT_DECAY
#         )
#         scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
#             optimizer, T_max=EPOCHS - FREEZE_EPOCHS
#         )
#         logger.info(f"  🔓 Epoch {epoch}: MLP unfrozen, LR reset to {LR_UNFROZEN}")

#     train_loss, train_f1, _, _          = run_epoch(train_loader, model, criterion, optimizer)
#     val_loss,   val_f1,  val_probs, val_labels = run_epoch(val_loader, model, criterion)
#     scheduler.step()

#     # Per-label F1 trên Val
#     per_label_f1 = f1_score(val_labels, (val_probs >= 0.5).astype(int),
#                              average=None, zero_division=0)
#     per_label_str = " | ".join(
#         f"{col.replace('_harm','')[:6]}={f1:.3f}"
#         for col, f1 in zip(HARM_COLS, per_label_f1)
#     )

#     logger.info(
#         f"Epoch {epoch:02d}/{EPOCHS} | "
#         f"Train Loss={train_loss:.4f} F1={train_f1:.4f} | "
#         f"Val Loss={val_loss:.4f} F1-Macro={val_f1:.4f}"
#     )
#     logger.info(f"  Per-label: {per_label_str}")

#     history.append({
#         "epoch": epoch,
#         "train_loss": train_loss, "train_f1_macro": train_f1,
#         "val_loss": val_loss,     "val_f1_macro": val_f1,
#         **{f"val_f1_{col}": float(f1) for col, f1 in zip(HARM_COLS, per_label_f1)},
#     })

#     # Checkpoint theo Val F1-Macro
#     if val_f1 > best_f1_macro:
#         best_f1_macro = val_f1
#         patience_cnt  = 0
#         # Tìm best threshold cho từng nhãn
#         best_ts = find_best_thresholds(val_labels, val_probs)
#         torch.save({
#             "epoch"      : epoch,
#             "model_state": model.state_dict(),
#             "val_f1_macro": val_f1,
#             "per_label_f1": per_label_f1.tolist(),
#             "best_thresholds": best_ts,
#         }, OUTPUT_DIR / "best_model.pt")
#         logger.info(f"  ✅ New best F1-Macro: {best_f1_macro:.4f} — saved best_model.pt")
#         logger.info(f"  Best thresholds: {dict(zip(HARM_COLS, best_ts))}")
#     else:
#         patience_cnt += 1
#         if patience_cnt >= PATIENCE:
#             logger.info(f"  ⏹️  Early stopping at epoch {epoch}")
#             break

# # Lưu checkpoint cuối
# torch.save({"epoch": epoch, "model_state": model.state_dict()},
#            OUTPUT_DIR / "last_model.pt")


## 4.7 Save

In [22]:
# pd.DataFrame(history).to_csv(OUTPUT_DIR / "train_history.csv", index=False)

# # Load best threshold từ best checkpoint
# best_ckpt = torch.load(OUTPUT_DIR / "best_model.pt", map_location="cpu")
# threshold_dict = dict(zip(HARM_COLS, best_ckpt["best_thresholds"]))
# with open(OUTPUT_DIR / "best_thresholds.json", "w") as f:
#     json.dump(threshold_dict, f, indent=2, ensure_ascii=False)

# logger.info("=" * 60)
# logger.info(f"✅ Stage 2 hoàn thành! Best Val F1-Macro: {best_f1_macro:.4f}")
# logger.info(f"Per-label F1 (best epoch): {dict(zip(HARM_COLS, best_ckpt['per_label_f1']))}")
# logger.info(f"Best thresholds: {threshold_dict}")
# logger.info("📦 NEXT STEPS:")
# logger.info("  1. Evaluate trên Test set bằng best_model.pt + best_thresholds.json")
# logger.info("  2. Visualize train_history.csv")
# logger.info("=" * 60)

# 5. Evaluation

## 5.1 Import

In [23]:
import sys

import numpy as np
import pandas as pd
import torch
import json
import logging
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    confusion_matrix, classification_report
)



logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("/kaggle/working/evaluation_run.log")
    ]
)
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
STAGE1_CKPT      = "/kaggle/input/datasets/kietkolat/stage-1-dataset/stage1_checkpoints/best_model.pt"
STAGE2_CKPT      = "/kaggle/input/datasets/kietkolat/stage-2-checkpoint/stage2_checkpoints/best_model.pt"
STAGE2_THRESHOLD = "/kaggle/input/datasets/kietkolat/stage-2-checkpoint/stage2_checkpoints/best_thresholds.json"

STAGE1_TEST_CSV  = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage1_test.csv"
STAGE2_TEST_CSV  = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/stage2_test.csv"
LABEL_CSV        = "/kaggle/input/datasets/kietkolat/split-data-dataset/data/splits/split_assignment.csv"

# Ngưỡng Stage 1 phân loại harmful (có thể điều chỉnh)
STAGE1_THRESHOLD = 0.5

OUTPUT_DIR  = Path("/kaggle/working/evaluation")
BATCH_SIZE  = 256   # Lớn hơn train vì chỉ inference, không backward
DROPOUT     = 0.3
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {DEVICE}")

2026-07-29 06:42:46,106 INFO Device: cuda


## 5.2 Load threshold of stage 2

In [24]:
with open(STAGE2_THRESHOLD, "r") as f:
    stage2_thresholds = json.load(f)  # {label: threshold}
threshold_array = np.array([stage2_thresholds[col] for col in HARM_COLS])
logger.info(f"Stage 2 thresholds: {stage2_thresholds}")

2026-07-29 06:42:46,131 INFO Stage 2 thresholds: {'information_harm': 0.65, 'sexual_harm': 0.85, 'psychological_harm': 0.3, 'hate_harassment_harm': 0.65, 'clickbait_harm': 0.55, 'addictive_harm': 0.85, 'physical_harm': 0.85}


## 5.3 Load model

In [25]:
stage2_model = FusionMLP(out_size=7, dropout=DROPOUT).to(DEVICE)
ckpt2 = torch.load(STAGE2_CKPT, map_location=DEVICE)
stage2_model.load_state_dict(ckpt2["model_state"])
stage2_model.eval()
logger.info(f"Stage 2 loaded (Val F1-Macro={ckpt2.get('val_f1_macro', '?')})")

2026-07-29 06:42:46,549 INFO Stage 2 loaded (Val F1-Macro=0.8492391105185385)


## 5.4 Inference helper

In [28]:
@torch.no_grad()
def run_inference(loader, model, device=DEVICE):
    """
    Chạy inference và trả về (probs, labels).
    probs: (N, out_size) numpy array
    labels: (N, out_size) numpy array
    """
    all_probs  = []
    all_labels = []

    for batch in tqdm(loader, desc="Inference", leave=False):
        vision  = batch["vision"].to(device)
        asr     = batch["asr"].to(device)
        ocr     = batch["ocr"].to(device)
        clap    = batch["clap"].to(device)
        tabular = batch["tabular"].to(device)
        labels  = batch["label"]

        logits = model(vision, asr, ocr, clap, tabular)
        probs  = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels.numpy())

    return np.vstack(all_probs), np.vstack(all_labels)

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1: Đánh giá Stage 1 trên test set
# ═══════════════════════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("📊 SECTION 1: STAGE 1 TEST EVALUATION")
logger.info("=" * 60)

stage1_test_loader = make_loader(
    STAGE1_TEST_CSV, mode="stage1",
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

s1_probs, s1_labels = run_inference(stage1_test_loader, stage1_model)
s1_probs  = s1_probs.flatten()
s1_labels = s1_labels.flatten().astype(int)
s1_preds  = (s1_probs >= STAGE1_THRESHOLD).astype(int)

s1_f1        = f1_score(s1_labels, s1_preds, zero_division=0)
s1_precision = precision_score(s1_labels, s1_preds, zero_division=0)
s1_recall    = recall_score(s1_labels, s1_preds, zero_division=0)
s1_roc_auc   = roc_auc_score(s1_labels, s1_probs)
s1_cm        = confusion_matrix(s1_labels, s1_preds)

logger.info(f"F1-Binary   : {s1_f1:.4f}")
logger.info(f"Precision   : {s1_precision:.4f}")
logger.info(f"Recall      : {s1_recall:.4f}")
logger.info(f"ROC-AUC     : {s1_roc_auc:.4f}")
logger.info(f"Confusion Matrix:\n{s1_cm}")

# Tìm threshold tối ưu cho Stage 1 trên test set
best_t1, best_f1_t1 = STAGE1_THRESHOLD, s1_f1
for t in np.arange(0.1, 0.9, 0.05):
    preds = (s1_probs >= t).astype(int)
    f1    = f1_score(s1_labels, preds, zero_division=0)
    if f1 > best_f1_t1:
        best_f1_t1, best_t1 = f1, t
logger.info(f"Best threshold (test) : {best_t1:.2f} → F1={best_f1_t1:.4f}")

# Save Stage 1 report
s1_report = {
    "threshold"    : float(STAGE1_THRESHOLD),
    "best_threshold_on_test": float(round(best_t1, 2)),
    "f1_binary"    : float(s1_f1),
    "precision"    : float(s1_precision),
    "recall"       : float(s1_recall),
    "roc_auc"      : float(s1_roc_auc),
    "confusion_matrix": s1_cm.tolist(),
    "n_test"       : int(len(s1_labels)),
    "n_harmful"    : int(s1_labels.sum()),
    "n_normal"     : int(len(s1_labels) - s1_labels.sum()),
}
with open(OUTPUT_DIR / "stage1_test_report.json", "w") as f:
    json.dump(s1_report, f, indent=2)

pd.DataFrame(
    s1_cm,
    index=["Actual Normal", "Actual Harmful"],
    columns=["Pred Normal", "Pred Harmful"]
).to_csv(OUTPUT_DIR / "stage1_confusion_matrix.csv")

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2: Đánh giá Stage 2 độc lập (chỉ trên harmful videos)
# ═══════════════════════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("📊 SECTION 2: STAGE 2 TEST EVALUATION (harmful only)")
logger.info("=" * 60)

stage2_test_loader = make_loader(
    STAGE2_TEST_CSV, mode="stage2",
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

s2_probs, s2_labels = run_inference(stage2_test_loader, stage2_model)
s2_labels = s2_labels.astype(int)
s2_preds  = (s2_probs >= threshold_array).astype(int)  # Per-label threshold

s2_f1_macro = f1_score(s2_labels, s2_preds, average="macro", zero_division=0)
s2_f1_micro = f1_score(s2_labels, s2_preds, average="micro", zero_division=0)
s2_per_label_f1 = f1_score(s2_labels, s2_preds, average=None, zero_division=0)

logger.info(f"F1-Macro    : {s2_f1_macro:.4f}")
logger.info(f"F1-Micro    : {s2_f1_micro:.4f}")
logger.info("Per-label F1:")
for col, f1 in zip(HARM_COLS, s2_per_label_f1):
    n_pos = s2_labels[:, HARM_COLS.index(col)].sum()
    logger.info(f"  {col:<30}: {f1:.4f}  (n_pos={n_pos})")

logger.info("\nClassification Report:")
logger.info(classification_report(
    s2_labels, s2_preds,
    target_names=[c.replace("_harm", "") for c in HARM_COLS],
    zero_division=0
))

# Save Stage 2 report
s2_report = {
    "thresholds_used" : stage2_thresholds,
    "f1_macro"        : float(s2_f1_macro),
    "f1_micro"        : float(s2_f1_micro),
    "per_label"       : {
        col: {
            "f1"       : float(f1),
            "precision": float(precision_score(s2_labels[:, i], s2_preds[:, i], zero_division=0)),
            "recall"   : float(recall_score(s2_labels[:, i], s2_preds[:, i], zero_division=0)),
            "n_pos_test": int(s2_labels[:, i].sum()),
        }
        for i, (col, f1) in enumerate(zip(HARM_COLS, s2_per_label_f1))
    },
    "n_test": int(len(s2_labels)),
}
with open(OUTPUT_DIR / "stage2_test_report.json", "w") as f:
    json.dump(s2_report, f, indent=2, ensure_ascii=False)

# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3: Cascade Inference (Stage1 → Stage2) trên toàn bộ test set
# ═══════════════════════════════════════════════════════════════════════════════
logger.info("=" * 60)
logger.info("📊 SECTION 3: CASCADE INFERENCE (End-to-End)")
logger.info("=" * 60)

# Load toàn bộ test set (dùng stage1_test.csv làm gốc — có đủ mọi video)
cascade_loader = make_loader(
    STAGE1_TEST_CSV, mode="stage1",
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

# Bước 1: Chạy Stage 1 trên toàn bộ test set
s1_probs_all, s1_labels_all = run_inference(cascade_loader, stage1_model)
s1_probs_all  = s1_probs_all.flatten()
s1_labels_all = s1_labels_all.flatten().astype(int)

harmful_mask = (s1_probs_all >= STAGE1_THRESHOLD)  # Boolean mask
n_predicted_harmful = harmful_mask.sum()
logger.info(f"Stage 1 predicted harmful: {n_predicted_harmful} / {len(s1_labels_all)}")

# Bước 2: Load stage2_test CSV để lấy ground truth 7 nhãn cho test set
# Join qua filename
stage1_df = pd.read_csv(STAGE1_TEST_CSV, dtype={"filename": str})
stage2_df = pd.read_csv(STAGE2_TEST_CSV, dtype={"filename": str})

# Merge để có ground truth 7 nhãn cho toàn bộ test set (normal = 0 cho tất cả nhãn)
full_test_df = stage1_df[["filename", "is_harmful"]].copy()
full_test_df = full_test_df.merge(
    stage2_df[["filename"] + HARM_COLS], on="filename", how="left"
)
full_test_df[HARM_COLS] = full_test_df[HARM_COLS].fillna(0).astype(int)
gt_labels = full_test_df[HARM_COLS].values  # (N, 7) ground truth

# Bước 3: Khởi tạo kết quả cascade = tất cả 0
cascade_preds = np.zeros((len(s1_labels_all), 7), dtype=int)

# Bước 4: Với các video Stage 1 predict là harmful → chạy Stage 2
# Cần tạo sub-loader cho chỉ những video đó
# Cách đơn giản: chạy Stage 2 trên stage2_test toàn bộ, rồi map lại
stage2_all_loader = make_loader(
    STAGE1_TEST_CSV, mode="stage2",
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

# Load Stage 2 model chạy trên toàn bộ test (kể cả normal)
# Chỉ apply kết quả vào những row mà Stage 1 predict harmful
with torch.no_grad():
    s2_probs_all = []
    for batch in tqdm(stage2_all_loader, desc="Stage 2 inference (all test)", leave=False):
        vision  = batch["vision"].to(DEVICE)
        asr     = batch["asr"].to(DEVICE)
        ocr     = batch["ocr"].to(DEVICE)
        clap    = batch["clap"].to(DEVICE)
        tabular = batch["tabular"].to(DEVICE)
        logits  = stage2_model(vision, asr, ocr, clap, tabular)
        probs   = torch.sigmoid(logits).cpu().numpy()
        s2_probs_all.append(probs)

s2_probs_all = np.vstack(s2_probs_all)  # (N, 7)

# Apply cascade: chỉ gán nhãn Stage 2 cho video được Stage 1 predict là harmful
cascade_preds[harmful_mask] = (
    s2_probs_all[harmful_mask] >= threshold_array
).astype(int)

# Đánh giá cascade
cascade_f1_macro = f1_score(gt_labels, cascade_preds, average="macro", zero_division=0)
cascade_f1_micro = f1_score(gt_labels, cascade_preds, average="micro", zero_division=0)
cascade_per_f1   = f1_score(gt_labels, cascade_preds, average=None, zero_division=0)

logger.info(f"Cascade F1-Macro : {cascade_f1_macro:.4f}")
logger.info(f"Cascade F1-Micro : {cascade_f1_micro:.4f}")
logger.info("Cascade Per-label F1:")
for col, f1 in zip(HARM_COLS, cascade_per_f1):
    logger.info(f"  {col:<30}: {f1:.4f}")

logger.info("\nCascade Classification Report:")
logger.info(classification_report(
    gt_labels, cascade_preds,
    target_names=[c.replace("_harm", "") for c in HARM_COLS],
    zero_division=0
))

# Save Cascade report
cascade_report = {
    "stage1_threshold" : float(STAGE1_THRESHOLD),
    "stage2_thresholds": stage2_thresholds,
    "n_test_total"     : int(len(s1_labels_all)),
    "n_predicted_harmful": int(n_predicted_harmful),
    "cascade_f1_macro" : float(cascade_f1_macro),
    "cascade_f1_micro" : float(cascade_f1_micro),
    "cascade_per_label": {
        col: float(f1) for col, f1 in zip(HARM_COLS, cascade_per_f1)
    },
}
with open(OUTPUT_DIR / "cascade_test_report.json", "w") as f:
    json.dump(cascade_report, f, indent=2, ensure_ascii=False)

2026-07-29 06:44:22,443 INFO ============================================================
2026-07-29 06:44:22,445 INFO 📊 SECTION 1: STAGE 1 TEST EVALUATION
2026-07-29 06:44:22,446 INFO ============================================================


NameError: name 'stage1_model' is not defined